In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("data/messy_data.csv")

In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")


In [4]:
df.replace(["Unknown", "", " "], np.nan, inplace=True)


In [5]:
# Fix 1: convert word-numbers to digits before coercing
word_to_num = {
    'zero':0,'one':1,'two':2,'three':3,'four':4,'five':5,'six':6,'seven':7,
    'eight':8,'nine':9,'ten':10,'eleven':11,'twelve':12,'thirteen':13,
    'fourteen':14,'fifteen':15,'sixteen':16,'seventeen':17,'eighteen':18,
    'nineteen':19,'twenty':20,'twenty-one':21,'twenty-two':22,'twenty-three':23,
    'twenty-four':24,'twenty-five':25,'twenty-six':26,'twenty-seven':27,
    'twenty-eight':28,'twenty-nine':29,'thirty':30,'thirty-one':31,
    'thirty-two':32,'thirty-three':33,'thirty-four':34,'thirty-five':35,
    'thirty-six':36,'thirty-seven':37,'thirty-eight':38,'thirty-nine':39,
    'forty':40,'forty-one':41,'forty-two':42,'forty-three':43,'forty-four':44,
    'forty-five':45,'forty-six':46,'forty-seven':47,'forty-eight':48,'forty-nine':49,
    'fifty':50,'fifty-one':51,'fifty-two':52,'fifty-three':53,'fifty-four':54,
    'fifty-five':55,'fifty-six':56,'fifty-seven':57,'fifty-eight':58,'fifty-nine':59,
    'sixty':60,'sixty-one':61,'sixty-two':62,'sixty-three':63,'sixty-four':64,
    'sixty-five':65,'sixty-six':66,'sixty-seven':67,'sixty-eight':68,'sixty-nine':69,
    'seventy':70,'seventy-five':75,'eighty':80,'eighty-five':85,'ninety':90,
}
df['age'] = (
    df['age'].astype(str).str.strip().str.lower()
    .map(lambda x: str(word_to_num.get(x, x)))
)
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['age'] = df['age'].fillna(df['age'].median())
df['age'] = df['age'].astype('Int64')

# Fix 2: use format='mixed' to handle all date formats correctly
df['visit_date'] = pd.to_datetime(df['visit_date'], format='mixed', dayfirst=False, errors='coerce')


In [6]:
df['gender'] = df['gender'].str.strip().str.lower()

# Fix 3: normalise gender abbreviations and variants
gender_map = {
    'm': 'male', 'man': 'male',
    'f': 'female', 'woman': 'female',
    'non-binary': 'other',
}
df['gender'] = df['gender'].replace(gender_map)

df['medication'] = df['medication'].str.strip().str.upper()


In [7]:
# Fix 4: map all 'no medication' variants to NONE, fill genuinely missing with NONE
df['medication'] = df['medication'].replace({
    'NO_MEDICATION': 'NONE',
    'N/A': 'NONE',
    'NA': 'NONE',
})
df['medication'] = df['medication'].fillna('NONE')


In [8]:
# Fix 5: normalise condition text (case + typos) — drop rows with missing label
condition_map = {
    'Asthma':'Asthma',        'asthma':'Asthma',        'ASTHMA':'Asthma',       'Athsma':'Asthma',
    'Diabetes':'Diabetes',    'diabetes':'Diabetes',    'DIABETES':'Diabetes',   'Diabets':'Diabetes',
    'Heart Disease':'Heart Disease', 'heart disease':'Heart Disease',
    'Heart disease':'Heart Disease', 'Hearth Disease':'Heart Disease',
    'Hypertension':'Hypertension', 'hypertension':'Hypertension',
    'HYPERTENSION':'Hypertension',  'Hypertention':'Hypertension',
}
df['condition'] = df['condition'].str.strip().map(
    lambda x: condition_map.get(x, x) if isinstance(x, str) else x
)
# Drop rows with missing condition — can't train a classifier without a label
df = df.dropna(subset=['condition'])


In [9]:
df = df.drop_duplicates()


In [10]:
# Fix 6: strip unit strings like '215 mg/dL', handle 'nan' string, fill missing
df['cholesterol'] = (
    df['cholesterol'].astype(str)
    .str.strip()
    .str.replace(r'[^\d.]', '', regex=True)  # remove 'mg/dL' and other non-numeric chars
    .replace({'': np.nan, 'nan': np.nan})     # catch both empty string and literal 'nan'
)
df['cholesterol'] = pd.to_numeric(df['cholesterol'], errors='coerce')
df['cholesterol'] = df['cholesterol'].fillna(df['cholesterol'].median())


In [11]:
df.drop(columns=["email", "phone_number"], inplace=True, errors="ignore")


In [12]:
df.reset_index(drop=True, inplace=True)


In [13]:
print(df.info())
print(df.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   patient_name    900 non-null    object        
 1   age             900 non-null    Int64         
 2   gender          900 non-null    object        
 3   condition       900 non-null    object        
 4   medication      900 non-null    object        
 5   visit_date      900 non-null    datetime64[ns]
 6   cholesterol     900 non-null    float64       
 7   blood_pressure  825 non-null    object        
dtypes: Int64(1), datetime64[ns](1), float64(1), object(5)
memory usage: 57.3+ KB
None
      patient_name  age  gender     condition    medication visit_date  \
0       donna hall   51  female      Diabetes  ATORVASTATIN 2021-06-19   
1       mark young   49    male      Diabetes     METFORMIN 2020-04-06   
2       lisa young   49    male  Hypertension          NONE 2018-05

In [14]:
df

,patient_name,age,gender,condition,medication,visit_date,cholesterol,blood_pressure
0,donna hall,51,female,Diabetes,ATORVASTATIN,2021-06-19,207.7,143/90
1,mark young,49,male,Diabetes,METFORMIN,2020-04-06,206.3,109 / 86
2,lisa young,49,male,Hypertension,NONE,2018-05-23,207.5,153/95
3,daniel robinson,46,male,Asthma,ALBUTEROL,2022-10-10,203.6,127/79
4,robert garcia,49,female,Hypertension,NONE,2018-05-10,199.5,161/93
...,...,...,...,...,...,...,...,...
895,james wilson,58,male,Diabetes,ATORVASTATIN,2020-09-10,228.3,NaN
896,donna martinez,49,female,Heart Disease,LISINOPRIL,2023-03-28,222.7,153/88
897,emily jackson,68,male,Diabetes,ATORVASTATIN,2019-02-21,213.8,153/80
898,robert walker,19,male,Asthma,NONE,2023-04-22,170.0,121/70


In [15]:
# Fix 7: strip spaces around '/' so '120 / 80' splits cleanly
df['blood_pressure'] = df['blood_pressure'].astype(str).str.replace(r'\s*/\s*', '/', regex=True)

df['systolic']  = df['blood_pressure'].str.split('/').str[0]
df['diastolic'] = df['blood_pressure'].str.split('/').str[1]

df['systolic']  = pd.to_numeric(df['systolic'],  errors='coerce')
df['diastolic'] = pd.to_numeric(df['diastolic'], errors='coerce')

df['systolic']  = df['systolic'].fillna(df['systolic'].median())
df['diastolic'] = df['diastolic'].fillna(df['diastolic'].median())

df.drop(columns=['blood_pressure'], inplace=True)


In [16]:
df

,patient_name,age,gender,condition,medication,visit_date,cholesterol,systolic,diastolic
0,donna hall,51,female,Diabetes,ATORVASTATIN,2021-06-19,207.7,143.0,90.0
1,mark young,49,male,Diabetes,METFORMIN,2020-04-06,206.3,109.0,86.0
2,lisa young,49,male,Hypertension,NONE,2018-05-23,207.5,153.0,95.0
3,daniel robinson,46,male,Asthma,ALBUTEROL,2022-10-10,203.6,127.0,79.0
4,robert garcia,49,female,Hypertension,NONE,2018-05-10,199.5,161.0,93.0
...,...,...,...,...,...,...,...,...,...
895,james wilson,58,male,Diabetes,ATORVASTATIN,2020-09-10,228.3,136.0,87.0
896,donna martinez,49,female,Heart Disease,LISINOPRIL,2023-03-28,222.7,153.0,88.0
897,emily jackson,68,male,Diabetes,ATORVASTATIN,2019-02-21,213.8,153.0,80.0
898,robert walker,19,male,Asthma,NONE,2023-04-22,170.0,121.0,70.0


In [17]:
# Cholesterol already cleaned and filled above (Fix 6)
# df['cholesterol'] = df['cholesterol'].fillna(df['cholesterol'].median())


In [18]:
df["visit_year"] = df["visit_date"].dt.year
df["visit_month"] = df["visit_date"].dt.month

df["visit_year"] = df["visit_year"].fillna(df["visit_year"].mode()[0])
df["visit_month"] = df["visit_month"].fillna(df["visit_month"].mode()[0])



In [19]:
df.to_csv('data/cleaned_data.csv', index=False)

In [20]:
df

,patient_name,age,gender,condition,medication,visit_date,cholesterol,systolic,diastolic,visit_year,visit_month
0,donna hall,51,female,Diabetes,ATORVASTATIN,2021-06-19,207.7,143.0,90.0,2021,6
1,mark young,49,male,Diabetes,METFORMIN,2020-04-06,206.3,109.0,86.0,2020,4
2,lisa young,49,male,Hypertension,NONE,2018-05-23,207.5,153.0,95.0,2018,5
3,daniel robinson,46,male,Asthma,ALBUTEROL,2022-10-10,203.6,127.0,79.0,2022,10
4,robert garcia,49,female,Hypertension,NONE,2018-05-10,199.5,161.0,93.0,2018,5
...,...,...,...,...,...,...,...,...,...,...,...
895,james wilson,58,male,Diabetes,ATORVASTATIN,2020-09-10,228.3,136.0,87.0,2020,9
896,donna martinez,49,female,Heart Disease,LISINOPRIL,2023-03-28,222.7,153.0,88.0,2023,3
897,emily jackson,68,male,Diabetes,ATORVASTATIN,2019-02-21,213.8,153.0,80.0,2019,2
898,robert walker,19,male,Asthma,NONE,2023-04-22,170.0,121.0,70.0,2023,4
